# LLaVA ScienceQA Replication
Replicating the ScienceQA evaluation from **"Visual Instruction Tuning" (LLaVA, NeurIPS 2023)**.

**Pipeline:**
1. Install dependencies
2. Load ScienceQA test split from HuggingFace
3. Run LLaVA (via LM Studio) on each question with image + answer choices
4. Use a Gemini judge to extract the predicted answer choice (A/B/C/D/E)
5. Compute accuracy against ground-truth answers

In [1]:
%pip install lmstudio datasets pillow requests google-generativeai -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import lmstudio as lms

# Verify LM Studio connection and show loaded models
with lms.Client() as client:
    models = client.llm.list_loaded()
    print("Connected to LM Studio!")
    print("Loaded models:", models)

Connected to LM Studio!
Loaded models: []


In [3]:

#Load the ScienceQA dataset and filter to image-based questions for evaluation


from datasets import load_dataset
from PIL import Image
import io, os, random

# Load ScienceQA test split from HuggingFace
# has_image=True filters to only multimodal questions (with images), matching the paper's eval setup
dataset = load_dataset("derek-thomas/ScienceQA", split="test")

# Filter to only questions that have an image (paper evaluates on image-based subset)
dataset_with_images = [s for s in dataset if s["image"] is not None]

# Sample 100 for a quick run; remove the slice for full eval
samples = dataset_with_images[:100]

print(f"Total test samples with images: {len(dataset_with_images)}")
print(f"Running evaluation on: {len(samples)} samples")
print()

# Show a sample entry
s = samples[0]
choices_str = " | ".join([f"{chr(65+i)}: {c}" for i, c in enumerate(s["choices"])])
print("Question:", s["question"])
print("Choices:", choices_str)
print("Answer:", chr(65 + s["answer"]), "->", s["choices"][s["answer"]])
print("Subject:", s.get("subject", "N/A"), "| Topic:", s.get("topic", "N/A"))

c:\Users\Kelvin\anaconda3\envs\llava\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total test samples with images: 2017
Running evaluation on: 100 samples

Question: Which of the following could Gordon's test show?
Choices: A: if the spacecraft was damaged when using a parachute with a 1 m vent going 200 km per hour | B: how steady a parachute with a 1 m vent was at 200 km per hour | C: whether a parachute with a 1 m vent would swing too much at 400 km per hour
Answer: B -> how steady a parachute with a 1 m vent was at 200 km per hour
Subject: natural science | Topic: science-and-engineering-practices


Run LLaVA Inference on ScienceQA

For each question, build a prompt matching the paper's format:
- Image (if available)
- Question text
- Answer choices (A, B, C, ...)
- Context / lecture / hint if present

LLaVA generates a free-form response that we can used to judge later


In [4]:
%pip install lmstudio datasets pillow requests -q


Note: you may need to restart the kernel to use updated packages.


In [5]:
from datasets import load_dataset
from PIL import Image
import base64, io, requests, os, random

# Load dataset and filter to image-based questions
dataset = load_dataset("derek-thomas/ScienceQA", split="test")
dataset_with_images = [s for s in dataset if s["image"] is not None]
samples = dataset_with_images[:100]
print(f"Loaded {len(samples)} samples.")

LMS_URL = "http://localhost:11434/v1/chat/completions"
LLAVA_MODEL = "llava:13b"

def pil_to_base64(pil_image: Image.Image) -> str:
    """Convert a PIL image to a base64-encoded PNG string."""
    buf = io.BytesIO()
    pil_image.convert("RGB").save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def build_llava_prompt(sample):
    """Build the multiple-choice prompt matching the LLaVA paper's ScienceQA format."""
    choices_str = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
    prompt = f"Question: {sample['question']}\n\nChoices:\n{choices_str}\n"
    if sample.get("hint"):
        prompt += f"\nHint: {sample['hint']}"
    if sample.get("lecture"):
        prompt += f"\nContext: {sample['lecture']}"
    prompt += "\n\nAnswer with the letter of the correct choice (A, B, C, ...) and a brief explanation."
    return prompt

def query_llava(sample):
    """Send a vision+text request to Ollama using OpenAI-compatible API."""
    img_b64 = pil_to_base64(sample["image"])
    payload = {
        "model": LLAVA_MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
                    {"type": "text", "text": build_llava_prompt(sample)}
                ]
            }
        ],
        "max_tokens": 512,
        "temperature": 0
    }
    resp = requests.post(LMS_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

llava_responses = []

for i, sample in enumerate(samples):
    response = query_llava(sample)
    llava_responses.append(response)
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(samples)}] {response[:80]}...")

print(f"\nDone. {len(llava_responses)} responses collected.")


Loaded 100 samples.
[10/100]  C. The strength of the magnetic force is the same in both pairs.

In the image,...
[20/100]  C. sticky...
[30/100]  B. It has many different types of organisms.

Yasuni National Park is part of t...
[40/100]  B...
[50/100]  B. Argema mittrei...
[60/100]  The correct answer is B. 2/4.

To determine the probability of an American curl...
[70/100]  A. Albuquerque...
[80/100]  C...
[90/100]  D...
[100/100]  A. Haiti...

Done. 100 responses collected.


In [6]:
import re, requests

# ── GPT Judge: Consensus / Tie-Breaker (matching the LLaVA paper) ──
# 1. LLaVA answers with image (already collected in llava_responses)
# 2. GPT (gpt-oss-20b) answers independently via text-only
# 3. If both agree → final answer
# 4. If they disagree → GPT acts as judge, reviewing both reasoning chains

JUDGE_URL = "http://localhost:1234/v1/chat/completions"
JUDGE_MODEL = "openai/gpt-oss-20b"

def extract_letter(text):
    """Extract a single answer letter (A-E) from a response."""
    text = text.strip().upper()
    m = re.search(r'^([A-E])[.\):\s]', text)
    if not m:
        m = re.search(r'ANSWER[:\s]*([A-E])', text)
    if not m:
        m = re.search(r'\(([A-E])\)', text)
    if not m:
        m = re.search(r'(?<![A-Z])([A-E])(?![A-Z])', text)
    return m.group(1) if m else "?"

def extract_from_reasoning_model(data):
    """Extract answer from a reasoning model response."""
    content = (data.get("content") or "").strip().upper()
    reasoning = (data.get("reasoning") or "").strip().upper()
    
    # 1. Content field has the final answer
    if content:
        m = re.search(r'[A-E]', content)
        if m:
            return m.group(0)
    
    # 2. Look for conclusion patterns in reasoning
    if reasoning:
        # "the answer is X", "correct answer is X", "I choose X", "therefore X"
        for pattern in [
            r'(?:THE\s+)?(?:CORRECT\s+)?ANSWER\s+IS\s+([A-E])',
            r'(?:I\s+)?CHOOSE\s+([A-E])',
            r'THEREFORE[,:]?\s*([A-E])',
            r'FINAL\s+ANSWER[:\s]+([A-E])',
            r'(?:SO|THUS)[,:]?\s*THE\s+ANSWER\s+IS\s+([A-E])',
        ]:
            m = re.search(pattern, reasoning)
            if m:
                return m.group(1)
        # Fallback: last standalone letter
        matches = re.findall(r'(?<![A-Z])([A-E])(?![A-Z])', reasoning)
        if matches:
            return matches[-1]
    
    return "?"

def gpt_answer(sample):
    """GPT answers the question independently (text-only, no image)."""
    choices_str = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
    prompt = f"Question: {sample['question']}\n\nChoices:\n{choices_str}\n"
    if sample.get("hint"):
        prompt += f"\nHint: {sample['hint']}"
    if sample.get("lecture"):
        prompt += f"\nContext: {sample['lecture']}"
    prompt += "\n\nAnswer with the letter of the correct choice and explain your reasoning."
    
    payload = {
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0
    }
    resp = requests.post(JUDGE_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]

def gpt_judge_tiebreak(sample, llava_response, gpt_response_text, llava_letter, gpt_letter):
    """GPT judge: given both reasoning chains, pick between the two answers."""
    choices_str = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
    prompt = (
        f"You are a judge for a multiple-choice science question.\n\n"
        f"Question: {sample['question']}\n\nChoices:\n{choices_str}\n"
    )
    if sample.get("hint"):
        prompt += f"\nHint: {sample['hint']}"
    if sample.get("lecture"):
        prompt += f"\nContext: {sample['lecture']}"
    prompt += (
        f"\n\n--- Assistant 1 (visual model with image access) chose {llava_letter} ---\n"
        f"{llava_response}\n\n"
        f"--- Assistant 2 (text-only model) chose {gpt_letter} ---\n"
        f"{gpt_response_text}\n\n"
        f"Assistant 1 chose {llava_letter} and Assistant 2 chose {gpt_letter}. "
        f"You MUST pick either {llava_letter} or {gpt_letter}. "
        f"Analyze both reasoning processes and determine which one is correct. "
        f"Reply with ONLY the letter {llava_letter} or {gpt_letter}."
    )
    payload = {
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0
    }
    resp = requests.post(JUDGE_URL, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()["choices"][0]["message"]
    result = extract_from_reasoning_model(data)
    
    # Constrain to only the two proposed answers
    if result not in (llava_letter, gpt_letter):
        # Count mentions of each in reasoning to decide
        full = (data.get("content") or "") + " " + (data.get("reasoning") or "")
        full = full.upper()
        llava_count = full.count(f"ASSISTANT 1") + full.count(llava_letter)
        gpt_count = full.count(f"ASSISTANT 2") + full.count(gpt_letter)
        # Default to whichever was mentioned more favorably; tie goes to LLaVA (has visual info)
        result = gpt_letter if gpt_count > llava_count else llava_letter
    
    return result

# ── Run the consensus pipeline ──
predicted_answers = []
gpt_responses = []
agree_count = 0
tiebreak_count = 0

for i, (sample, llava_resp) in enumerate(zip(samples, llava_responses)):
    try:
        # Step 1: Extract LLaVA's answer
        llava_letter = extract_letter(llava_resp)
        
        # Step 2: GPT answers independently (text-only)
        gpt_msg = gpt_answer(sample)
        gpt_text = (gpt_msg.get("content") or "") + " " + (gpt_msg.get("reasoning") or "")
        gpt_responses.append(gpt_text)
        gpt_letter = extract_from_reasoning_model(gpt_msg)
        
        # Step 3: Consensus or tie-break
        if llava_letter == gpt_letter and llava_letter != "?":
            final = llava_letter
            agree_count += 1
        else:
            final = gpt_judge_tiebreak(sample, llava_resp, gpt_text, llava_letter, gpt_letter)
            tiebreak_count += 1
            
    except Exception as e:
        print(f"  ERROR on sample {i}: {e}")
        final = extract_letter(llava_resp)
    
    predicted_answers.append(final)
    
    if (i + 1) % 10 == 0:
        gt = chr(65 + sample["answer"])
        print(f"[{i+1}/{len(samples)}] LLaVA={llava_letter} GPT={gpt_letter} Final={final} GT={gt}")

print(f"\nDone. {len(predicted_answers)} predictions.")
print(f"Agreed: {agree_count} | Tie-breaks: {tiebreak_count}")
print(f"Unparsed (?): {predicted_answers.count('?')}/{len(predicted_answers)}")


[10/100] LLaVA=C GPT=A Final=A GT=B
[20/100] LLaVA=C GPT=A Final=C GT=A
[30/100] LLaVA=B GPT=B Final=B GT=B
[40/100] LLaVA=B GPT=A Final=B GT=B
[50/100] LLaVA=B GPT=A Final=B GT=B
[60/100] LLaVA=B GPT=A Final=A GT=C
[70/100] LLaVA=A GPT=B Final=B GT=B
[80/100] LLaVA=C GPT=E Final=C GT=C
[90/100] LLaVA=D GPT=A Final=D GT=D
[100/100] LLaVA=A GPT=A Final=A GT=A

Done. 100 predictions.
Agreed: 46 | Tie-breaks: 54
Unparsed (?): 0/100


## Step 5: Compute Accuracy

The paper reports LLaVA achieves **90.92%** accuracy on ScienceQA (GPT-4 as judge), surpassing GPT-4 alone (87.56%).

In [8]:
from collections import defaultdict
import re

correct = 0
llava_correct = 0
gpt_correct = 0
subject_stats = defaultdict(lambda: {"correct": 0, "total": 0})

results = []
for i, (sample, pred) in enumerate(zip(samples, predicted_answers)):
    gt = chr(65 + sample["answer"])
    is_correct = (pred == gt)
    correct += is_correct

    # Also score LLaVA-only and GPT-only for comparison
    def extract_letter_simple(text):
        text = (text or "").strip().upper()
        m = re.search(r'^([A-E])[.\):\s]', text)
        if not m: m = re.search(r'ANSWER[:\s]*([A-E])', text)
        if not m: m = re.search(r'(?<![A-Z])([A-E])(?![A-Z])', text)
        return m.group(1) if m else "?"

    llava_pred = extract_letter_simple(llava_responses[i])
    gpt_pred = extract_letter_simple(gpt_responses[i]) if i < len(gpt_responses) else "?"
    llava_correct += (llava_pred == gt)
    gpt_correct += (gpt_pred == gt)

    subject = sample.get("subject", "unknown")
    subject_stats[subject]["correct"] += is_correct
    subject_stats[subject]["total"] += 1

    results.append({
        "id": i,
        "question": sample["question"][:60] + "...",
        "ground_truth": gt,
        "predicted": pred,
        "llava_only": llava_pred,
        "gpt_only": gpt_pred,
        "correct": is_correct,
        "subject": subject
    })

n = len(samples)
overall_accuracy = correct / n * 100
print(f"{'='*50}")
print(f"  Final (LLaVA + GPT judge):  {correct}/{n} = {overall_accuracy:.2f}%")
print(f"  LLaVA only:                 {llava_correct}/{n} = {llava_correct/n*100:.2f}%")
print(f"  GPT only (text-only):       {gpt_correct}/{n} = {gpt_correct/n*100:.2f}%")
print(f"  Paper (LLaVA-13B + GPT-4):  90.92%")
print(f"{'='*50}")
print()
print("Accuracy by subject:")
for subject, stats in sorted(subject_stats.items()):
    acc = stats["correct"] / stats["total"] * 100
    print(f"  {subject:<25} {stats['correct']:>3}/{stats['total']:<3} = {acc:.1f}%")


  Final (LLaVA + GPT judge):  74/100 = 74.00%
  LLaVA only:                 70/100 = 70.00%
  GPT only (text-only):       77/100 = 77.00%
  Paper (LLaVA-13B + GPT-4):  90.92%

Accuracy by subject:
  language science            1/1   = 100.0%
  natural science            47/64  = 73.4%
  social science             26/35  = 74.3%
